In [1]:
from utils import (
    read_csv_file,
    clean_row,
    build_chunks,
    save_chunks,
    build_dong_lookup,
    save_json,
)

1. Load Korean Waste Policies from CSV File

In [ ]:
waste_data = read_csv_file(file_path="../Data/korean_waste_policies.csv")
print(len(waste_data), "rows loaded")

Loaded with encoding=cp949, sep=',', 10187 rows
10187 rows loaded


2. Clean Every Rows

In [3]:
records = [clean_row(row) for row in waste_data.to_dict(orient="records")]
print(records[0])   # check one record

{'id': 'waste_202030000000400002', 'province': '서울특별시', 'district': '종로구', 'zone': '', 'dongs': [], 'general_method': '규격봉투에 넣어 지정된 요일에 배출', 'general_days': '일, 월, 화, 수, 목, 금', 'general_time': '19:00~21:00', 'food_method': '물기를 최대한 줄여 음식물만 전용봉투에 넣어 배출', 'food_days': '일, 월, 화, 수, 목, 금', 'food_time': '19:00~21:00', 'recycle_method': '투명한 비닐봉투에 담거나 끈으로 묶어서 지정된 요일에 배출', 'recycle_days': '일, 월, 화, 수, 목, 금', 'recycle_time': '19:00~21:00', 'bulky_method': '주민센터방문 혹은 종로구홈페이지에 신청 후 배출', 'bulky_place': '집앞', 'no_collection': '토, 구정연휴, 추석연휴', 'department': '청소행정과', 'phone': '02-2148-2373', 'data_date': '2020-07-10'}


3. Build Chunks

### How I build Chunks
- It takes one cleaned record (one row) and joins the useful fields into one multi-line text string. That string becomes the text of one chunk.
- I am concated most of columns information for text for better information provide to retrieval.

In [4]:
chunks = build_chunks(records)
print(chunks[0]["text"])   # check one chunk

[서울특별시 종로구] 쓰레기 배출 안내
생활쓰레기: 규격봉투에 넣어 지정된 요일에 배출 / 요일: 일, 월, 화, 수, 목, 금 / 시간: 19:00~21:00
음식물쓰레기: 물기를 최대한 줄여 음식물만 전용봉투에 넣어 배출 / 요일: 일, 월, 화, 수, 목, 금 / 시간: 19:00~21:00
재활용품: 투명한 비닐봉투에 담거나 끈으로 묶어서 지정된 요일에 배출 / 요일: 일, 월, 화, 수, 목, 금 / 시간: 19:00~21:00
대형폐기물: 주민센터방문 혹은 종로구홈페이지에 신청 후 배출 / 장소: 집앞
미수거일: 토, 구정연휴, 추석연휴
문의: 청소행정과 (02-2148-2373)
데이터 기준일: 2020-07-10


### Explain Information in Chunk in Json
```
├── id
├── text
└── metadata
    ├── district
    ├── zone
    ├── dongs
    ├── phone
    └── data_date
```

- ID: A unique name for this chunk
- Text: The full content of the rule, written as readable Korean sentences.
- metadata: It's not use for embedding, just for filtering and display only
    - district: example “Jongno-gu”, "종로구"
    - zone: A sub-area inside a district that has its own schedule. It is empty when the district has only one rule.
    - dongs: The neighborhoods (동) covered by this zone
    - phone: Showing the contact example (“Call 02-2148-2373 to confirm”)
    - data_date: The date the original record was made.


4. Save Chunks

In [5]:
save_chunks(chunks, "output/chunks.jsonl")

Saved 10187 chunks to output/chunks.jsonl


5. Build and save the dong lookup

In [6]:
lookup = build_dong_lookup(records)
save_json(lookup, "output/dong_lookup.json")

Saved output/dong_lookup.json
